In [5]:
import argparse
import re
import sys
import traceback
from pathlib import Path
import stim

#  file anme: q_Q_n_N_k_K_d_D
FNAME_RE = re.compile(r"q[_-]?(\d+)[_-]+n[_-]?(\d+)[_-]+k[_-]?(\d+)[_-]+d[_-]?(\d+)", re.I)
OUT = Path("circuits")
OUT.mkdir(parents = True, exist_ok = True)
PAULI = {(0, 0): "_", (1, 0): "X", (0, 1): "Z", (1, 1): "Y"}

In [26]:
ROWS = """
[1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 1 0 0 1 1 0 0 1 0 0 1 0 0 0 0 0 0|0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 1 1 0 0 1 1 0 1 1 1 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 1 1 0 0 1 1 0 1 1 1 0 0 0 0 0 0|1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0]
      [0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 1 0 1 1 0 0 1 1 0 1 0 0 0 0 0 0 0|0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 0 0 1 0 1 0 1 0 0 1 0 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 0 0 1 0 1 0 1 0 0 1 0 0 0 0 0 0 0|0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 1 1 0 0 1 0 0 0 0 0 0 0 0 0]
      [0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 1 0 1 1 0 0 1 1 0 1 0 0 0 0 0 0|0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 0 0 1 0 1 0 1 0 0 1 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 0 0 1 0 1 0 1 0 0 1 0 0 0 0 0 0|0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 1 1 0 0 1 0 0 0 0 0 0 0 0]
      [0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 1 0 0 1 1 0 0 0 0 0 0 0 0 0|0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 0 1 0 1 0 0 1 1 1 0 1 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 0 1 0 1 0 0 1 1 1 0 1 0 0 0 0 0 0|0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0]
      [0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 1 0 1 1 1 1 0 1 1 0 0 0 0 0 0|0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 1 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 1 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0|0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 1 1 1 0 1 1 1 1 0 0 1 0 1 1 0 0 0 0 0 0]
      [0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 1 0 1 0 0 0 0 1 1 1 0 1 0 0 0 0 0 0 0 0|0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 1 1 0 1 1 1 1 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 1 1 0 1 1 1 1 0 0 0 0 0 0|0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 1 0 0 0 1 0 0 0 0 1 1 0 1 1 0 0 0 0 0 0]
      [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 1 1 1 0 0 0 0 1 1 0 1 0 0 0 0 0 0|0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 1 0 1 0 1 1 0 0 1 0 0 1 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 1 0 1 0 1 1 0 0 1 0 0 1 0 0 0 0 0 0|0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 1 0 1 0 1 1 1 0 0 0 1 0 0 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 1 1 0 0 0 0 1 1 1 1 0 0 0 0 0 0 0 0 0|0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 1 1 0 1 0 1 1 0 1 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 1 1 0 1 0 1 1 0 1 0 0 0 0 0 0|0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 1 1 1 0 1 0 1 0 1 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 1 1 1 0 0 0 0 1 0 1 1 0 0 0 0 0 0|0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 0 1 1 0 0 1 0 1 0 0 0 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 0 1 1 0 0 1 0 1 0 0 0 0 0 0 0 0 0|0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 1 0 1 1 0 0 0 0 1 0 0 0 1 1 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 1 0 1 1 1 1 0 1 0 0 1 1 0 0 0 0 0 0 0 0|0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 1 0 0 0 1 1 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 1 0 0 0 1 1 0 0 0 0 0 0|0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 1 0 1 0 0 1 1 1 1 0 1 1 1 1 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 1 0 0 1 0 0 1 0 0 0 1 0 0 0 0 0 0|0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 1 1 0 0 1 0 1 0 1 1 1 1 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 1 1 0 0 1 0 1 0 1 1 1 1 0 0 0 0 0 0|0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 1 0 0 0 0 0 0 1 1 1 1 1 0 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 1 0 0 1 1 0 1 1 0 1 1 0 0 0 0 0 0 0|0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 0 1 0 0 1 1 1 1 0 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 0 1 0 0 1 1 1 1 0 0 0 0 0 0 0|0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 1 1 0 0 1 0 1 0 0 0 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 1 0 0 1 1 0 1 1 0 1 1 0 0 0 0 0 0|0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 0 1 0 0 1 1 1 1 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 0 1 0 0 1 1 1 1 0 0 0 0 0 0|0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 1 1 0 0 1 0 1 0 0 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 1 0 0 1 1 0 0 1 0 0 1 1 0 0 0 0 0 0|0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 1 1 0 0 1 1 0 1 1 1 0 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 1 1 0 0 1 1 0 1 1 1 0 0 0 0 0 0 0|0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 1 1 1 1 1 1 1 1 1 1 1 0 1 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0|0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0|0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0|0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0|0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0|0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
      [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1|0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
""".replace("[","").replace("]","").strip().splitlines()


35 1 11


In [ ]:
# sanity check
N = len(ROWS[0].replace(" ","").split("|")[0])
k = N - len(ROWS)
d = None # ADD distance here from the website -> or write code to brute force find it idk
print(N, k, d)

In [ ]:
def parse():
    gens = []
    for row in ROWS:
        xpart, zpart = row.split("|")
        x = [int(b) for b in xpart.split()]
        z = [int(b) for b in zpart.split()]
        assert len(x) == N and len(z) == N
        gens.append(stim.PauliString("+" + "".join(PAULI[(x[j], z[j])] for j in range(N))))
    return gens

def mpp_targets(ps):
    tgt = []
    # p in {0:I, 1:X, 2:Y, 3:Z}
    for q, p in enumerate(ps):                 
        if p == 0:
            continue
        f = {1: stim.target_x, 2: stim.target_y, 3: stim.target_z}[p]
        if tgt:
            tgt.append(stim.target_combiner())
        # a negative-sign product flips its outcome: invert exactly one target
        tgt.append(f(q, invert=(ps.sign == -1 and not tgt)))
    return tgt

def build_mpp_directly(gens):
    c = stim.Circuit()
    for ps in gens:
        c.append("MPP", mpp_targets(ps))
    return c

def build_mpp_from_tableau(gens):
    tab = stim.Tableau.from_stabilizers(gens, allow_redundant = False, allow_underconstrained = True)
    m = len(gens)
    c = stim.Circuit()
    for k in range(m):
        c.append("MPP", mpp_targets(tab.z_output(k)))
    return c, tab

def build_ancilla(gens):
    c = stim.Circuit()
    anc = {i: N + i for i in range(len(gens))}
    c.append("R", list(anc.values()))
    c.append("H", list(anc.values()))
    for i, s in enumerate(gens):
        a = anc[i]
        for q, p in enumerate(s):
            if p == 1:   c.append("CX", [a, q])
            elif p == 2: c.append("CY", [a, q])
            elif p == 3: c.append("CZ", [a, q])
    c.append("H", list(anc.values()))
    c.append("M", list(anc.values()))
    return c

In [ ]:
gens = parse()
m = len(gens)

# SANITY CHECKS
print(f"{m} generators on {N} qubits encoding {k} qubits")

ok = all(gens[i].commutes(gens[j]) for i in range(m) for j in range(i + 1, m))
tab_chk = stim.Tableau.from_stabilizers(gens, allow_redundant = False, allow_underconstrained = True)
print("pairwise commute:", ok, "| independent & valid:", True)

mpp_direct = build_mpp_directly(gens)
mpp_tab, tab = build_mpp_from_tableau(gens)
print("Route A (direct) == Route B (tableau):", mpp_direct == mpp_tab)
# SANITY CHECKS END

ancilla = build_ancilla(gens)
encoder = tab.to_circuit(method="graph_state")

# files saved -> most important is the syndorome_mpp_[...].stim -> rest are just there
files = { 
    f"syndrome_mpp_{N}_{k}_{d}.stim": str(mpp_direct),
    f"syndrome_mpp_from_tableau_{N}_{k}_{d}.stim": str(mpp_tab),
    f"syndrome_ancilla_{N}_{k}_{d}.stim": str(ancilla),
    f"encoder_{N}_{k}_{d}.stim": str(encoder),
    f"stabilizers_{N}_{k}_{d}.txt": "".join(f"g{i:2d}: {s}\n" for i, s in enumerate(gens, 1))
}

for name, text in files.items():
    (OUT / name).write_text(text)
print("saved to", OUT)